# Short-Term Memory — LangGraph

A chat graph whose `messages` list (persisted per `thread_id` via a
`checkpointer`) IS the short-term memory. On every turn, before
calling the model, the node applies two strategies so the running
history doesn't blow past the context window:

- **`trim_messages`** — keep only the LAST messages that fit under a
  max token budget (counted APPROXIMATELY, so no extra model call is
  needed just to measure length).
- **Summarization** — instead of just dropping whatever got trimmed
  away, compress it into a short summary so that context isn't lost
  outright.

In [ ]:
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, trim_messages
from langchain_core.messages.utils import count_tokens_approximately
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

load_dotenv()

model = ChatGroq(model="llama-3.1-8b-instant")

In [ ]:
class ChatState(TypedDict):
    # add_messages appends every new message onto this list — this list,
    # persisted per thread_id by the checkpointer below, IS the graph's
    # short-term memory.
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def summarize_dropped_messages(all_messages, kept_messages) -> SystemMessage | None:
    kept_ids = {id(m) for m in kept_messages}
    dropped = [
        m
        for m in all_messages
        if id(m) not in kept_ids and not isinstance(m, SystemMessage)
    ]

    if not dropped:
        return None

    conversation = "\n".join(f"{m.type}: {m.content}" for m in dropped)
    summary_prompt = (
        "Summarize the following conversation history in 1-2 sentences, "
        f"keeping only information that might be useful later:\n\n{conversation}"
    )
    summary = model.invoke(summary_prompt).content

    return SystemMessage(content=f"Summary of earlier conversation: {summary}")

In [ ]:
def chat_node(state: ChatState) -> ChatState:
    messages = state["messages"]

    # trim_messages — keep only the LAST messages that fit under a max
    # token budget (counted approximately), instead of replaying the
    # entire thread history into the context window on every turn.
    trimmed_messages = trim_messages(
        messages,
        strategy="last",
        token_counter=count_tokens_approximately,
        max_tokens=60,
        start_on="human",
        include_system=True,
    )

    # summarization — compress whatever got trimmed away into a short
    # summary instead of losing it outright.
    summary_message = summarize_dropped_messages(messages, trimmed_messages)

    system_messages = [m for m in trimmed_messages if isinstance(m, SystemMessage)]
    recent_messages = [m for m in trimmed_messages if not isinstance(m, SystemMessage)]

    final_messages = (
        system_messages
        + ([summary_message] if summary_message else [])
        + recent_messages
    )

    response = model.invoke(final_messages)
    return {"messages": [response]}

In [ ]:
# checkpointer — required so `state["messages"]` persists across
# separate invoke() calls on the same thread_id.
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
# print/visualize the graph
from IPython.display import Image, display

display(Image(workflow.get_graph().draw_mermaid_png()))

## Run a multi-turn conversation on one `thread_id`

Each call only sends the new `HumanMessage` — the checkpointer supplies
the rest of `state["messages"]` from the previous turns on this thread.

In [ ]:
config = {"configurable": {"thread_id": "1"}}

turns = [
    "Hi, I'm Apeksha.",
    "What's the capital of France?",
    "And what about Germany?",
    "Thanks! One more — what's the capital of Italy?",
]

for turn in turns:
    result = workflow.invoke(
        {"messages": [HumanMessage(content=turn)]}, config=config
    )
    print("user:", turn)
    print("assistant:", result["messages"][-1].content, "\n")